# scRareBench | High-Level Harmony | Dataset 4 — COVID-19 autoimmunity PBMC

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This notebook runs **Harmony** on scRareBench Dataset 4 through the high-level API:

`load_dataset(4) → MethodSpec(Harmony) → benchmark_method() → scIB + rare benchmark → HTML / multi-seed bundle`

Pinned scRareBench commit: `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`.

Dataset 4 contract:
- 97,499 cells × 21,858 features
- label: `cell_type` (evaluation only; never passed to Harmony)
- batch: `donor_id`
- canonical counts: `layers["counts"]`
- scIB HVG mode: `evaluation_batch`
- 22 donors / 26 biological label categories
- registered rare-scenario metadata is package-owned

Method-side preprocessing:
- donor-aware Seurat-v3 4,000 HVGs, span 0.3 (`batch_key="donor_id"`)
- normalize_total 10,000 → log1p → scale(max_value=10)
- PCA 50
- Harmony 2.0.0 on `donor_id`

Dataset 4 does not need Dataset 3's global-HVG numerical workaround; donor-aware
Seurat-v3 HVG selection passed the Dataset 4 audit.

Use a **High-RAM** Colab runtime. Harmony itself does not require a GPU.
For a first smoke run, temporarily set `METHOD_SEEDS = [42]`.

## 1. Install pinned scRareBench + Harmony

In [ ]:
import subprocess, sys

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
SCRAREBENCH_GITHUB = (
    "git+https://github.com/amirhossein-alishahi/"
    f"scRareBench_.git@{SCRAREBENCH_REVISION}"
)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    SCRAREBENCH_GITHUB,
])

METHOD_DEPENDENCIES = ("harmonypy==2.0.0",)

from scrarebench.runtime import setup_runtime, print_install_report

report = setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES,
    extra_imports=("harmonypy",),
    quiet=False,
)
print_install_report(report)

subprocess.check_call([
    sys.executable,
    "-c",
    "import harmonypy as hm; print('harmonypy', hm.__version__)",
])


## 2. Load Dataset 4

In [ ]:
import scrarebench
from scrarebench import dataset_info, load_dataset, scenario_table_from_adata

adata = load_dataset(4)
info = dataset_info(adata)

expected = {
    "dataset_key": "covid19_autoimmunity_pbmc",
    "label_key": "cell_type",
    "batch_key": "donor_id",
    "count_layer": "counts",
    "scib_hvg_batch_mode": "evaluation_batch",
    "benchmark_ready": True,
}
for key, value in expected.items():
    if info.get(key) != value:
        raise RuntimeError(
            f"Unexpected Dataset 4 contract for {key!r}: "
            f"observed={info.get(key)!r}, expected={value!r}"
        )

if adata.shape != (97_499, 21_858):
    raise RuntimeError(
        f"Unexpected Dataset 4 shape: {adata.shape}; expected (97499, 21858)."
    )
if "counts" not in adata.layers:
    raise RuntimeError("Dataset 4 canonical raw-count layer 'counts' is missing.")
if not adata.obs_names.is_unique:
    raise RuntimeError("Dataset 4 cell identifiers are not unique.")
if not adata.var_names.is_unique:
    raise RuntimeError("Dataset 4 feature identifiers are not unique.")
if adata.obs["donor_id"].astype(str).nunique() != 22:
    raise RuntimeError("Unexpected Dataset 4 donor cardinality; expected 22.")
if adata.obs["cell_type"].astype(str).nunique() != 26:
    raise RuntimeError("Unexpected Dataset 4 label cardinality; expected 26.")

scenario_table = scenario_table_from_adata(adata)
if scenario_table is None or scenario_table.empty:
    raise RuntimeError("Dataset 4 registered rare-scenario metadata is missing.")
if len(scenario_table) != 12:
    raise RuntimeError(
        f"Unexpected Dataset 4 scenario-table size: {len(scenario_table)}; expected 12."
    )

print("scRareBench:", scrarebench.__version__)
print(adata)
print("Donors:", adata.obs["donor_id"].astype(str).nunique())
print("Cell-type categories:", adata.obs["cell_type"].astype(str).nunique())
print("Registered six-state rows:", len(scenario_table))
print(info)


## 3. Harmony configuration

In [ ]:
METHOD_SEEDS = [42, 123, 2026]   # use [42] for a first smoke run
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "n_hvg": 4000,
    "hvg_span": 0.3,
    "hvg_batch_key": "donor_id",
    "normalize_target_sum": 10_000.0,
    "scale_max_value": 10.0,
    "n_pcs": 50,
    "pca_solver": "arpack",

    "theta": 2.0,
    "lamb": None,
    "sigma": 0.1,
    "nclust": None,
    "tau": 0.0,
    "block_size": 0.05,
    "max_iter_harmony": 10,
    "max_iter_kmeans": 4,
    "epsilon_cluster": 1e-3,
    "epsilon_harmony": 1e-2,
    "alpha": 0.2,
    "batch_prop_cutoff": 1e-5,
    "ncores": 0,
    "verbose": True,
}

OUTPUT_DIR = "/content/scrarebench_harmony_dataset4_highlevel"
METHOD_WORK_DIR = "/content/scrarebench_harmony_dataset4_method_runs"


## 4. User-owned Harmony runner

Dataset-specific preparation remains inside scRareBench. This runner only performs
Harmony's method-owned preprocessing/integration. Biological labels are not used.

For Dataset 4, method-side HVGs are donor-aware (`batch_key="donor_id"`) using
Seurat-v3, 4,000 HVGs and span 0.3.


In [ ]:
from pathlib import Path
import gc
import shutil

import anndata as ad
import harmonypy as hm
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

from scrarebench import MethodOutput, dataset_info


def run_harmony(method_adata, seed, config):
    method_info = dataset_info(method_adata)
    batch_key = method_info["batch_key"]
    count_layer = method_info["count_layer"]

    if method_info.get("dataset_key") != "covid19_autoimmunity_pbmc":
        raise RuntimeError("This runner is configured only for scRareBench Dataset 4.")
    if batch_key != "donor_id":
        raise RuntimeError(
            f"Unexpected Dataset 4 batch key {batch_key!r}; expected 'donor_id'."
        )
    if method_info.get("scib_hvg_batch_mode") != "evaluation_batch":
        raise RuntimeError(
            "Dataset 4 is expected to use evaluation-batch-aware HVGs."
        )
    if not count_layer or count_layer not in method_adata.layers:
        raise KeyError(f"Missing canonical raw-count layer: {count_layer!r}")
    if batch_key not in method_adata.obs.columns:
        raise KeyError(f"Missing Harmony batch key: {batch_key!r}")

    hvg_batch_key = str(config["hvg_batch_key"])
    if hvg_batch_key != batch_key:
        raise RuntimeError(
            f"Dataset 4 Harmony HVGs must use donor_id: "
            f"hvg_batch_key={hvg_batch_key!r}, batch_key={batch_key!r}."
        )

    original_barcodes = method_adata.obs_names.astype(str).to_numpy(copy=True)
    if len(np.unique(original_barcodes)) != len(original_barcodes):
        raise RuntimeError("Benchmark cell identifiers must be unique.")

    # Dataset 4 method-side HVG policy: donor-aware Seurat-v3 4000 HVGs.
    sc.pp.highly_variable_genes(
        method_adata,
        layer=count_layer,
        flavor="seurat_v3",
        n_top_genes=min(int(config["n_hvg"]), method_adata.n_vars),
        batch_key=hvg_batch_key,
        span=float(config["hvg_span"]),
        subset=False,
        check_values=True,
    )

    if "highly_variable" not in method_adata.var.columns:
        raise RuntimeError("Seurat-v3 HVG selection did not return a mask.")

    mask = method_adata.var["highly_variable"].fillna(False).to_numpy(dtype=bool)
    expected_hvg = min(int(config["n_hvg"]), method_adata.n_vars)
    if int(mask.sum()) != expected_hvg:
        raise RuntimeError(
            f"Expected {expected_hvg} HVGs, selected {int(mask.sum())}."
        )

    counts = method_adata.layers[count_layer][:, mask]
    counts = (
        counts.tocsr().astype(np.float32)
        if sp.issparse(counts)
        else np.asarray(counts, dtype=np.float32)
    )

    work = ad.AnnData(
        X=counts,
        obs=method_adata.obs.copy(),
        var=method_adata.var.loc[method_adata.var_names[mask]].copy(),
    )

    if not work.var_names.is_unique:
        raise RuntimeError("Selected Dataset 4 HVG identifiers are non-unique.")
    if not np.array_equal(work.obs_names.astype(str).to_numpy(), original_barcodes):
        raise RuntimeError("Cell identity/order changed before Harmony.")

    sc.pp.normalize_total(
        work,
        target_sum=float(config["normalize_target_sum"]),
    )
    sc.pp.log1p(work)
    sc.pp.scale(work, max_value=float(config["scale_max_value"]))

    n_pcs = min(
        int(config["n_pcs"]),
        work.n_vars - 1,
        work.n_obs - 1,
    )
    if n_pcs < 2:
        raise RuntimeError(f"Too few PCA dimensions: {n_pcs}")

    sc.pp.pca(
        work,
        n_comps=n_pcs,
        zero_center=True,
        svd_solver=str(config["pca_solver"]),
        random_state=int(seed),
    )

    pca = np.asarray(work.obsm["X_pca"], dtype=np.float64)
    expected_pca_shape = (work.n_obs, n_pcs)
    if pca.shape != expected_pca_shape:
        raise RuntimeError(
            f"Unexpected PCA shape: {pca.shape}; expected {expected_pca_shape}."
        )
    if not np.isfinite(pca).all():
        raise ValueError("PCA contains NaN or infinite values.")

    np.random.seed(int(seed))
    ho = hm.run_harmony(
        pca,
        work.obs,
        batch_key,
        theta=float(config["theta"]),
        lamb=config["lamb"],
        sigma=float(config["sigma"]),
        nclust=config["nclust"],
        tau=float(config["tau"]),
        block_size=float(config["block_size"]),
        max_iter_harmony=int(config["max_iter_harmony"]),
        max_iter_kmeans=int(config["max_iter_kmeans"]),
        epsilon_cluster=float(config["epsilon_cluster"]),
        epsilon_harmony=float(config["epsilon_harmony"]),
        alpha=float(config["alpha"]),
        batch_prop_cutoff=float(config["batch_prop_cutoff"]),
        verbose=bool(config["verbose"]),
        random_state=int(seed),
        ncores=int(config["ncores"]),
    )

    latent = np.asarray(ho.Z_corr, dtype=np.float32)

    if latent.shape != pca.shape:
        raise RuntimeError(
            f"Unexpected Harmony latent shape: {latent.shape}; expected {pca.shape}."
        )
    if not np.isfinite(latent).all():
        raise ValueError("Harmony latent contains NaN or infinite values.")
    if not np.array_equal(work.obs_names.astype(str).to_numpy(), original_barcodes):
        raise RuntimeError("Cell identity/order changed during Harmony.")

    seed_dir = Path(METHOD_WORK_DIR) / f"seed_{int(seed)}"
    if seed_dir.exists():
        shutil.rmtree(seed_dir)
    seed_dir.mkdir(parents=True, exist_ok=True)

    provenance = {}
    objective = np.asarray(
        getattr(ho, "objective_harmony", []),
        dtype=float,
    ).reshape(-1)

    if objective.size:
        objective_path = seed_dir / "harmony_objective_history.csv"
        pd.DataFrame({
            "harmony_iteration": np.arange(objective.size, dtype=int),
            "objective_harmony": objective,
        }).to_csv(objective_path, index=False)
        provenance["harmony_objective_history"] = objective_path

    print(
        f"Seed {seed}: Harmony latent={latent.shape}; "
        f"HVGs={work.n_vars}; "
        f"batches={work.obs[batch_key].astype(str).nunique()}"
    )

    del ho, work, counts, pca
    gc.collect()

    return MethodOutput(
        latent=latent,
        barcodes=original_barcodes,
        representation_key="X_pca_harmony",
        provenance_files=provenance,
    )


## 5. Run the high-level benchmark

In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="Harmony",
    runner=run_harmony,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        # Dataset 4 metadata supplies scib_hvg_batch_mode="evaluation_batch".
        "write_pdf_report": False,
        "include_latent_in_bundle": False,
        "include_cell_ids": False,
    },
    install_dependencies=False,
    finalize=True,
    copy_input_per_seed=True,
)


## 6. Inspect results

In [ ]:
from IPython.display import display

display(result.summary().round(5))
print("Method:", result.method)
print("Seeds:", result.seeds)
print("Multi-seed HTML:", result.report_path)
print("Archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(f"\nSeed {seed}")
    print("  representation:", run.representation_key)
    print("  alignment:", run.alignment)
    print("  report:", run.interactive_report_path)
    print("  bundle:", run.bundle_path)

    if run.representation_key != "X_pca_harmony":
        raise RuntimeError(
            f"Seed {seed} returned an unexpected representation key."
        )
    if run.alignment.get("exact_order_match") is not True:
        raise RuntimeError(
            f"Seed {seed} did not preserve exact barcode order."
        )


## 7. Download final archive

In [ ]:
from pathlib import Path

if result.archive_path is None:
    raise RuntimeError(
        "No multi-seed archive was created. Run all final seeds with finalize=True."
    )

archive = Path(result.archive_path)
if not archive.exists():
    raise FileNotFoundError(archive)

print("Archive:", archive)
print("Size (MB):", round(archive.stat().st_size / 1024 / 1024, 2))

try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print("Not running in Colab; archive remains at:", archive)
